In [1]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import jax
import jax.numpy as jnp

from utils.datatype import constructStructuredEvarray

width, height = 240, 180
sequence = "dynamic_rotation"
base_dir = f"data/Event Camera Dataset/{sequence}"
txyp_array = np.loadtxt(f"{base_dir}/events.txt")
calib = np.loadtxt(f"{base_dir}/calib.txt")
fx, fy, cx, cy, *dist = calib
cmx = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
dist = np.array(dist)
if len(dist) not in [4, 5, 8, 12, 14]:
    dist = np.r_[dist, np.zeros(4 - len(dist))]
imu = np.loadtxt(f"{base_dir}/imu.txt")
groundtruth = np.loadtxt(f"{base_dir}/groundtruth.txt")

undistortedxy_map = cv2.undistortPoints(
    np.mgrid[:width, :height].reshape(2, -1).astype(float), cmx, dist
).reshape(width, height, 2)

undistortedxy_map = undistortedxy_map.reshape(width, height, 2)
mean_xy = undistortedxy_map.mean(axis=(0, 1))

undistorted_evarray = constructStructuredEvarray(
    txyp_array[:, 0],
    *undistortedxy_map[txyp_array[:, 1].astype(int), txyp_array[:, 2].astype(int)].T,
    txyp_array[:, 3],
    np.ones(len(txyp_array)),
)

In [2]:
import functional.numpy.stat_jax as fstat
from scipy.optimize import minimize
from utils.warp import warp3drot_linapprox as warp3drot

warpbins = np.array([200, 150])
warpranges = jnp.array(
    [
        [-1.0 + mean_xy[0], 1.0 + mean_xy[0]],
        [-0.75 + mean_xy[1], 0.75 + mean_xy[1]],
    ]
)
jhist2d = jax.jit(fstat.jhist2d_fixedrange_linear(warpbins, warpranges))


@jax.jit
def warp3dofHist(rotvec: jax.Array, txy_array: jax.Array, weights: jax.Array):
    return jhist2d(warp3drot(rotvec, txy_array, txy_array[:, 0].mean()), weights)


@jax.jit
def warp3dofNegContrast(rotvec: jax.Array, txy_array: jax.Array, weights: jax.Array):
    return -warp3dofHist(rotvec, txy_array, weights).var()


vagfun = jax.jit(jax.value_and_grad(warp3dofNegContrast))
hessfun = jax.jit(jax.hessian(warp3dofNegContrast))

In [ ]:
from scipy.interpolate import CubicSpline
from jax.scipy.spatial.transform import Rotation
from tqdm import tqdm
from utils.analysis import rmse
from IPython.display import display, clear_output

allrots = []
allts = []
alltm = []
allte = []
imgs = []
x0 = jnp.zeros(3)
nevents = 20000
fig, ax = plt.subplots()
for evarray in tqdm(
    np.array_split(undistorted_evarray, len(undistorted_evarray) // nevents)
):
    txy_array = jnp.c_[evarray["t"], evarray["x"], evarray["y"]]
    weights = jnp.ones(len(evarray))
    optres = minimize(
        vagfun,
        x0,
        args=(txy_array, weights),
        jac=True,
        hess=hessfun,
        method="trust-ncg",
    )
    x0 = optres.x
    ax.clear()
    imgs.append(warp3dofHist(x0, txy_array, weights).T)
    ax.imshow(imgs[-1])
    clear_output(wait=True)
    display(fig)

    allrots.append(-x0)
    allts.append(txy_array[0, 0])
    alltm.append(txy_array[:, 0].mean())
    allte.append(txy_array[-1, 0])

plt.close(fig)
allrots = jnp.stack(allrots)
allts = jnp.stack(allts)
alltm = jnp.stack(alltm)
allte = jnp.stack(allte)
imgs = np.stack(imgs)
plt.plot(allte, allrots)
allkeyts = jnp.r_[allts[0], allte]
drot = (allkeyts[1:] - allkeyts[:-1])[:, None] * allrots
absdrots = Rotation.from_rotvec(drot)

imugen = CubicSpline(
    imu[:, 0],
    imu[:, -3:],
)
gtrotvecs = imugen(alltm)

errinc = gtrotvecs - allrots
print(rmse(errinc[:], alltm[:]))